## 1. Setup

In [1]:
import torch
import torch.nn.functional as F

def time_kernel(fn, *args, n_warmup=10, n_iters=100):
    for _ in range(n_warmup): fn(*args)
    torch.cuda.synchronize()
    start = torch.cuda.Event(enable_timing=True); end = torch.cuda.Event(enable_timing=True)
    start.record()
    for _ in range(n_iters): fn(*args)
    end.record(); torch.cuda.synchronize()
    return start.elapsed_time(end) / n_iters

print("torch:", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))

torch: 2.10.0+cu128 | GPU: Tesla T4


## 2. The eager-execution problem

When PyTorch evaluates `layer_norm(x + residual, ...)`, it launches multiple separate kernels, each round-tripping a `(B, S, D)` tensor through HBM:

So bandwidth is wasted on the intermediate IO.

In [2]:
B, S, D = 16, 1024, 4096
x        = torch.randn(B, S, D, device='cuda')
residual = torch.randn(B, S, D, device='cuda')
weight   = torch.randn(D, device='cuda')
bias     = torch.randn(D, device='cuda')

def eager_ln_residual(x, residual, weight, bias):
    return F.layer_norm(x + residual, (D,), weight, bias)

ms_eager = time_kernel(eager_ln_residual, x, residual, weight, bias)
print(f"PyTorch eager: {ms_eager:.3f} ms")

# Approximate HBM bytes moved:
#   read x, read residual                       -> 2 * B*S*D * 4
#   write (x+res), read it back for layernorm   -> 2 * B*S*D * 4
#   write output                                -> 1 * B*S*D * 4
bytes_moved = 5 * B*S*D * 4
print(f"  effective bandwidth: {bytes_moved/(ms_eager*1e-3)/1e9:.0f} GB/s")

PyTorch eager: 6.572 ms
  effective bandwidth: 204 GB/s


## 3. Install Triton

In [3]:
!pip install -q triton
import triton
import triton.language as tl
print("triton:", triton.__version__)

triton: 3.6.0


In [4]:
@triton.jit
def fused_ln_residual_kernel(
    X_ptr, R_ptr, W_ptr, B_ptr, Y_ptr,
    stride_row, N_COLS,
    eps,
    BLOCK_SIZE: tl.constexpr,
):
    row = tl.program_id(0)
    cols = tl.arange(0, BLOCK_SIZE)
    mask = cols < N_COLS

    # Read x&r
    x = tl.load(X_ptr + row*stride_row + cols, mask=mask, other=0.0).to(tl.float32)
    r = tl.load(R_ptr + row*stride_row + cols, mask=mask, other=0.0).to(tl.float32)
    z = x + r

    # Mean and variance in one row pass
    mean = tl.sum(z, axis=0) / N_COLS
    diff = tl.where(mask, z - mean, 0.0)
    var  = tl.sum(diff * diff, axis=0) / N_COLS
    inv  = 1.0 / tl.sqrt(var + eps)

    # read W&B
    w = tl.load(W_ptr + cols, mask=mask, other=1.0)
    b = tl.load(B_ptr + cols, mask=mask, other=0.0)
    y = (z - mean) * inv * w + b

    # write back to HBM
    tl.store(Y_ptr + row*stride_row + cols, y, mask=mask)


def fused_ln_residual(x, residual, weight, bias, eps=1e-5):
    assert x.shape == residual.shape
    y = torch.empty_like(x)
    n_rows = x.numel() // x.shape[-1]
    n_cols = x.shape[-1]
    BLOCK = triton.next_power_of_2(n_cols)
    fused_ln_residual_kernel[(n_rows,)](
        x.reshape(-1, n_cols), residual.reshape(-1, n_cols),
        weight, bias, y.reshape(-1, n_cols),
        n_cols, n_cols, eps, BLOCK_SIZE=BLOCK,
    )
    return y

In [5]:
y_fused = fused_ln_residual(x, residual, weight, bias)
y_eager = eager_ln_residual(x, residual, weight, bias)
print("max abs diff:", (y_fused - y_eager).abs().max().item())
assert torch.allclose(y_fused, y_eager, atol=1e-2, rtol=1e-2)

max abs diff: 2.86102294921875e-06


In [6]:
ms_fused = time_kernel(fused_ln_residual, x, residual, weight, bias)
print(f"PyTorch eager : {ms_eager:.3f} ms")
print(f"Fused (Triton): {ms_fused:.3f} ms")
print(f"Speedup       : {ms_eager / ms_fused:.2f}×")
print()
bytes_fused = 3 * B*S*D * 4
print(f"Fused bandwidth: {bytes_fused/(ms_fused*1e-3)/1e9:.0f} GB/s  "
      f"(of {320 if 'T4' in torch.cuda.get_device_name(0) else '???'} GB/s peak)")

PyTorch eager : 6.572 ms
Fused (Triton): 3.299 ms
Speedup       : 1.99×

Fused bandwidth: 244 GB/s  (of 320 GB/s peak)
